# Imports and Data Load

In [3]:
!pip install seqeval
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=aadf8cfd8bcc4a42c8182a52013a0a24ad2fc93decbf98ccb4485e11643c41f4
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.2 MB/s eta 0:00:00


In [4]:
import pandas as pd
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
from seqeval.metrics import classification_report as seqeval_classification_report
from sklearn.metrics import log_loss
import gensim.downloader as api
import numpy as np
from sklearn.linear_model import SGDClassifier
import torch
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os

os.environ["WANDB_DISABLED"] = "true"

In [5]:
df = pd.read_csv('https://drive.google.com/uc?id=1L-YG1btZHsH3LXOirNspv9k8cENUizoa', encoding ='latin1')

# Check structure of data

In [6]:
print(df.head())
print(df.isna().sum())
print(df['Tag'].value_counts())

    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O
Sentence #    1000616
Word               10
POS                 0
Tag                 0
dtype: int64
Tag
O        887908
B-geo     37644
B-tim     20333
B-org     20143
I-per     17251
B-per     16990
I-org     16784
B-gpe     15870
I-geo      7414
I-tim      6528
B-art       402
B-eve       308
I-art       297
I-eve       253
B-nat       201
I-gpe       198
I-nat        51
Name: count, dtype: int64


# Fill null values

In [7]:
df = df.ffill()

# Grouping words and tags by sentence

In [8]:
sentences=[]
labels=[]

grouped=df.groupby('Sentence #')

for _,group in grouped:
    words=list(group['Word'])
    tags=list(group['Tag'])
    sentences.append(words)
    labels.append(tags)

# Encode Labels (NER Tags)

In [9]:
le=LabelEncoder()

# Flatten all the tags into one big list
all_tags=list(set(tag for seq in labels for tag in seq))
le.fit(all_tags)

# Build two dictionaries to map back and forth
label2id = {tag: idx for idx, tag in enumerate(le.classes_)}
id2label = {idx: tag for tag, idx in label2id.items()}

# Tokenization & Label Alignment

In [10]:
def tokenize_and_align_labels(sentences, labels, tokenizer):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    aligned_labels = []
    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                # You could use the same label or 'I-' version
                label_ids.append(label2id[label[word_idx]])
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

# Creating a Dataset Class

In [11]:
class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

# Split train and validation data

In [12]:
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(
    sentences, labels, test_size=0.1, random_state=42
)

# Load Word2Vec

In [13]:
word2vec_model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


# Loading the Model to Start training

### Shallow Learning (Logistic Regression)

In [14]:
def get_word_vector(word, model):
    try:
        if hasattr(model, 'wv'):
            return model.wv[word]
        else:
            return model[word]
    except KeyError:
        return np.zeros(model.vector_size if hasattr(model, 'vector_size') else 300)

def prepare_data_for_sgd(sentences, labels, word2vec_model):
    X = []
    y = []
    count = 0
    for sent, tags in zip(sentences, labels):
        for word, tag in zip(sent, tags):
            word_vec = get_word_vector(word, word2vec_model)
            X.append(word_vec)
            y.append(label2id[tag])
            count += 1

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

X_train, y_train = prepare_data_for_sgd(train_sentences, train_labels, word2vec_model)
X_val, y_val = prepare_data_for_sgd(val_sentences, val_labels, word2vec_model)

sgd_model = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=0.0001,
    max_iter=1000,
    tol=1e-3,
    random_state=42,
    verbose=1,
    n_jobs=-1,
    learning_rate='optimal',
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5
)

sgd_model.fit(X_train, y_train)

y_val_pred_proba = sgd_model.predict_proba(X_val)
val_loss = log_loss(y_val, y_val_pred_proba)
print("Validation loss:", val_loss)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.


-- Epoch 1
-- Epoch 1
Norm: 2.12, NNZs: 300, Bias: -7.979559, T: 849636, Avg. loss: 0.002857
Total training time: 2.97 seconds.
-- Epoch 2
Norm: 3.05, NNZs: 300, Bias: -9.983249, T: 849636, Avg. loss: 0.002642
Total training time: 3.46 seconds.
-- Epoch 2
Norm: 2.16, NNZs: 300, Bias: -7.935602, T: 1699272, Avg. loss: 0.002824
Total training time: 6.83 seconds.
-- Epoch 3
Norm: 2.83, NNZs: 300, Bias: -9.158017, T: 1699272, Avg. loss: 0.001952
Total training time: 7.29 seconds.
-- Epoch 3
Norm: 2.17, NNZs: 300, Bias: -7.921191, T: 2548908, Avg. loss: 0.002818
Total training time: 9.93 seconds.
-- Epoch 4
Norm: 2.74, NNZs: 300, Bias: -8.847380, T: 2548908, Avg. loss: 0.001931
Total training time: 10.35 seconds.
-- Epoch 4
Norm: 2.19, NNZs: 300, Bias: -7.908940, T: 3398544, Avg. loss: 0.002807
Total training time: 12.95 seconds.
-- Epoch 5
Norm: 2.69, NNZs: 300, Bias: -8.720267, T: 3398544, Avg. loss: 0.001912
Total training time: 13.63 seconds.
-- Epoch 5
Norm: 2.19, NNZs: 300, Bias: -7.9

[Parallel(n_jobs=-1)]: Done  17 out of  17 | elapsed:  1.9min finished


### Deep Learning (Bi-LSTM)

In [ ]:
embedding_dim = 300

def get_word_vector(word, model, embedding_dim=300):
    if word in model.key_to_index:
        return model[word]
    else:
        return np.zeros(embedding_dim)

def prepare_sequences(sentences, labels, model):
    X, y = [], []
    for sent, tag_seq in zip(sentences, labels):
        word_vectors = [get_word_vector(w, model) for w in sent]
        tag_ids = [label2id[t] for t in tag_seq]
        X.append(np.array(word_vectors))
        y.append(np.array(tag_ids))
    return X, y

X_train, y_train = prepare_sequences(train_sentences, train_labels, word2vec_model)
X_val, y_val = prepare_sequences(val_sentences, val_labels, word2vec_model)

lengths = [len(seq) for seq in X_train + X_val]
max_len = int(np.percentile(lengths, 95))

X_train_padded = pad_sequences(X_train, maxlen=max_len, dtype='float32', padding='post', value=0.0)
X_val_padded = pad_sequences(X_val, maxlen=max_len, dtype='float32', padding='post', value=0.0)
y_train_padded = pad_sequences(y_train, maxlen=max_len, dtype='int32', padding='post', value=-100)
y_val_padded = pad_sequences(y_val, maxlen=max_len, dtype='int32', padding='post', value=-100)

num_labels = len(label2id)
hidden_dim = 64

def masked_sparse_categorical_crossentropy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.cast(tf.not_equal(y_true, -100), dtype=tf.float32)
    y_true_masked = tf.where(tf.equal(y_true, -100), tf.constant(0, dtype=tf.int32), y_true)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true_masked, y_pred)
    loss = loss * mask
    return tf.reduce_sum(loss) / (tf.reduce_sum(mask) + 1e-10)

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.cast(tf.not_equal(y_true, -100), dtype=tf.float32)
    y_pred_labels = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
    y_true_masked = tf.where(tf.equal(y_true, -100), tf.constant(0, dtype=tf.int32), y_true)
    correct = tf.cast(tf.equal(y_true_masked, y_pred_labels), dtype=tf.float32)
    correct = correct * mask
    return tf.reduce_sum(correct) / (tf.reduce_sum(mask) + 1e-10)

model_lstm = keras.Sequential([
    layers.Input(shape=(max_len, embedding_dim)),
    layers.Masking(mask_value=0.0),
    layers.LSTM(hidden_dim, return_sequences=True, dropout=0.3),
    layers.TimeDistributed(layers.Dense(num_labels, activation='softmax'))
], name='Simple_LSTM_NER')

model_bilstm = keras.Sequential([
    layers.Input(shape=(max_len, embedding_dim)),
    layers.Masking(mask_value=0.0),
    layers.Bidirectional(layers.LSTM(hidden_dim, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)),
    layers.TimeDistributed(layers.Dense(num_labels, activation='softmax'))
], name='Bidirectional_LSTM_NER')

model_bilstm.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=masked_sparse_categorical_crossentropy,
    metrics=[masked_accuracy]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history = model_bilstm.fit(
    X_train_padded, y_train_padded,
    validation_data=(X_val_padded, y_val_padded),
    epochs=10,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 43s 28ms/step - loss: 0.8625 - masked_accuracy: 0.9026 - val_loss: 0.3680 - val_masked_accuracy: 0.9523
Epoch 2/10
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 47s 35ms/step - loss: 0.3187 - masked_accuracy: 0.9529 - val_loss: 0.2149 - val_masked_accuracy: 0.9565
Epoch 3/10
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 53s 39ms/step - loss: 0.2059 - masked_accuracy: 0.9562 - val_loss: 0.1804 - val_masked_accuracy: 0.9578
Epoch 4/10
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 45s 33ms/step - loss: 0.1778 - masked_accuracy: 0.9576 - val_loss: 0.1655 - val_masked_accuracy: 0.9592
Epoch 5/10
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - loss: 0.1609 - masked_accuracy: 0.9600 - val_loss: 0.1597 - val_masked_accuracy: 0.9605
Epoch 6/10
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - loss: 0.1576 - masked_accuracy: 0.9602 - val_loss: 0.1563 - val_masked_accuracy: 0.9608
Epoch 7/10
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - loss: 0.1532 - masked_accuracy: 0.9609 - val_loss: 0.1548 - val_ma

### Performance (ALBERT)

In [ ]:
tokenizer_albert = AutoTokenizer.from_pretrained('albert-base-v2')
model_albert = AutoModelForTokenClassification.from_pretrained(
    'albert-base-v2',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    from_tf=False
)

train_encodings_albert = tokenize_and_align_labels(train_sentences, train_labels, tokenizer_albert)
val_encodings_albert = tokenize_and_align_labels(val_sentences, val_labels, tokenizer_albert)

train_dataset_albert = NERDataset(train_encodings_albert)
val_dataset_albert = NERDataset(val_encodings_albert)

data_collator_albert = DataCollatorForTokenClassification(tokenizer_albert)

training_args_albert = TrainingArguments(
    output_dir="./results_albert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    eval_strategy="epoch"
)

trainer_albert = Trainer(
    model=model_albert,
    args=training_args_albert,
    train_dataset=train_dataset_albert,
    eval_dataset=val_dataset_albert,
    tokenizer=tokenizer_albert,
    data_collator=data_collator_albert
)

trainer_albert.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Some weights of AlbertForTokenClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-3837738538.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_albert = Trainer(


Epoch,Training Loss,Validation Loss
1,0.141200,0.133693
2,0.113800,0.123518
3,0.090700,0.121651


TrainOutput(global_step=8094, training_loss=0.12489123069889492, metrics={'train_runtime': 4087.5739, 'train_samples_per_second': 31.679, 'train_steps_per_second': 1.98, 'total_flos': 1012538779439742.0, 'train_loss': 0.12489123069889492, 'epoch': 3.0})

### Efficient (DeBERTa)

In [ ]:
tokenizer_deberta = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
model_deberta = AutoModelForTokenClassification.from_pretrained(
    'microsoft/deberta-v3-base',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    from_tf=False
)

train_encodings_deberta = tokenize_and_align_labels(train_sentences, train_labels, tokenizer_deberta)
val_encodings_deberta = tokenize_and_align_labels(val_sentences, val_labels, tokenizer_deberta)

train_dataset_deberta = NERDataset(train_encodings_deberta)
val_dataset_deberta = NERDataset(val_encodings_deberta)

data_collator_deberta = DataCollatorForTokenClassification(tokenizer_deberta)

training_args_deberta = TrainingArguments(
    output_dir="./results_deberta",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    eval_strategy="epoch"
)

trainer_deberta = Trainer(
    model=model_deberta,
    args=training_args_deberta,
    train_dataset=train_dataset_deberta,
    eval_dataset=val_dataset_deberta,
    tokenizer=tokenizer_deberta,
    data_collator=data_collator_deberta
)

trainer_deberta.train()

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-2406239124.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_deberta = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordi

Epoch,Training Loss,Validation Loss
1,0.125400,0.116389
2,0.101500,0.107723
3,0.088800,0.105687


TrainOutput(global_step=8094, training_loss=0.11614249174996073, metrics={'train_runtime': 5289.1387, 'train_samples_per_second': 24.482, 'train_steps_per_second': 1.53, 'total_flos': 9517571788724064.0, 'train_loss': 0.11614249174996073, 'epoch': 3.0})

### Multilingual (XLM-R)

In [11]:
tokenizer_xlmr = AutoTokenizer.from_pretrained('xlm-roberta-base')
model_xlmr = AutoModelForTokenClassification.from_pretrained(
    'xlm-roberta-base',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    from_tf=False
)

train_encodings_xlmr = tokenize_and_align_labels(train_sentences, train_labels, tokenizer_xlmr)
val_encodings_xlmr = tokenize_and_align_labels(val_sentences, val_labels, tokenizer_xlmr)

train_dataset_xlmr = NERDataset(train_encodings_xlmr)
val_dataset_xlmr = NERDataset(val_encodings_xlmr)

data_collator_xlmr = DataCollatorForTokenClassification(tokenizer_xlmr)

training_args_xlmr = TrainingArguments(
    output_dir="./results_xlmr",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    eval_strategy="epoch"
)

trainer_xlmr = Trainer(
    model=model_xlmr,
    args=training_args_xlmr,
    train_dataset=train_dataset_xlmr,
    eval_dataset=val_dataset_xlmr,
    tokenizer=tokenizer_xlmr,
    data_collator=data_collator_xlmr
)

trainer_xlmr.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-3161543807.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_xlmr = Trainer(


Epoch,Training Loss,Validation Loss
1,0.121900,0.114583
2,0.099000,0.106312


Epoch,Training Loss,Validation Loss
1,0.121900,0.114583
2,0.099000,0.106312
3,0.084700,0.104621


TrainOutput(global_step=8094, training_loss=0.11591090453769767, metrics={'train_runtime': 4332.1119, 'train_samples_per_second': 29.891, 'train_steps_per_second': 1.868, 'total_flos': 1.0310516605300392e+16, 'train_loss': 0.11591090453769767, 'epoch': 3.0})

### Monolingual (RoBERTa)

In [27]:
tokenizer_roberta = AutoTokenizer.from_pretrained('roberta-base', add_prefix_space=True)

model_roberta = AutoModelForTokenClassification.from_pretrained(
    'roberta-base',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    from_tf=False
)

train_encodings_roberta = tokenize_and_align_labels(train_sentences, train_labels, tokenizer_roberta)
val_encodings_roberta = tokenize_and_align_labels(val_sentences, val_labels, tokenizer_roberta)

train_dataset_roberta = NERDataset(train_encodings_roberta)
val_dataset_roberta = NERDataset(val_encodings_roberta)

data_collator_roberta = DataCollatorForTokenClassification(tokenizer_roberta)

training_args_roberta = TrainingArguments(
    output_dir="./results_roberta",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=1,
    eval_strategy="epoch"
)

trainer_roberta = Trainer(
    model=model_roberta,
    args=training_args_roberta,
    train_dataset=train_dataset_roberta,
    eval_dataset=val_dataset_roberta,
    tokenizer=tokenizer_roberta,
    data_collator=data_collator_roberta
)

trainer_roberta.train()

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-2869148826.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_roberta = Trainer(


Epoch,Training Loss,Validation Loss
1,0.138500,0.130592
2,0.112900,0.121442


Epoch,Training Loss,Validation Loss
1,0.138500,0.130592
2,0.112900,0.121442
3,0.094700,0.120782


TrainOutput(global_step=8094, training_loss=0.1274149590490599, metrics={'train_runtime': 3875.9835, 'train_samples_per_second': 33.408, 'train_steps_per_second': 2.088, 'total_flos': 9980051329489482.0, 'train_loss': 0.1274149590490599, 'epoch': 3.0})